In [1]:
import tsl
import torch

In [2]:
from tsl.datasets import MetrLA

dataset = MetrLA(root='./data')


c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\tsl\datasets\metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\tsl\datasets\metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


In [3]:
dataset.shape

(34272, 207, 1)

In [4]:
df = dataset.dataframe()

df.head()

nodes,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
channels,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,62.750000,...,45.625000,65.500,64.500000,66.428574,66.875,59.375000,69.000000,59.250000,69.000000,61.875
2012-03-01 00:05:00,62.666668,68.555557,65.444443,62.444443,64.444443,68.111115,65.000,65.000,57.444443,63.333332,...,50.666668,69.875,66.666664,58.555557,62.000,61.111111,64.444443,55.888889,68.444443,62.875
2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857140,62.000
2012-03-01 00:15:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857140,62.000
2012-03-01 00:20:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857140,62.000


In [5]:
from tsl.data import SpatioTemporalDataset

# Number of edges depend on the threshold

connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        normalize_axis=1,
                                        layout="edge_index")

torch_dataset = SpatioTemporalDataset(target=df,
                                      connectivity=connectivity, # Do not add it and use learnable params
                                      mask=dataset.mask,
                                      horizon=12,
                                      window=12,
                                      stride=1)

In [6]:
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler # Dont need it to add standard scaler in csv preprocessing

# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.3)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
)

In [7]:


from lib.nn.encoders.corel_encoder import CoRelEncoder
from lib.nn.decoder.base_decoder import BaseDecoder
from lib.nn.encoder_decoder_model import EncoderDecoderModel


In [8]:
def print_model_size(model):
    tot = sum([p.numel() for p in model.parameters() if p.requires_grad])
    out = f"Number of model ({model.__class__.__name__}) parameters:{tot:10d}"
    print("=" * len(out))
    print(out)

hidden_size = 32   #@param
temporal_layers = 1     #@param
spatial_layers = 1     #@param
conv_type = "diffconv"  #@param ["diffconv", "graphconv"]
temporal_type = "gru"   #@param ["gru", "lstm"]

input_size = torch_dataset.n_channels   # 1 channel
n_nodes = torch_dataset.n_nodes         # 207 nodes
horizon = torch_dataset.horizon         # 12 time steps


stgnn = EncoderDecoderModel(
    input_size=input_size,
    output_size=input_size,
    horizon=horizon,
    encoder_class=CoRelEncoder,
    encoder_kwargs={'gnn_layers': spatial_layers, 'temporal_layers': temporal_layers,
                    'hidden_size': hidden_size, 'n_instances': n_nodes, 'emb_size': hidden_size,
                    'n_neighbors':2, 'conv_type': conv_type,},
    decoder_class=BaseDecoder,
    decoder_kwargs={},
    exog_size= 0,
)




print_model_size(stgnn)

TypeError: GRUCellBase.__init__() got an unexpected keyword argument 'input_size'

In [ ]:
from tsl.metrics.torch import MaskedMAE, MaskedMAPE
from tsl.engines import Predictor

loss_fn = MaskedMAE()

metrics = {'mae': MaskedMAE(),
           'mape': MaskedMAPE(),
           'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step,
                                          # which correspond to 15 minutes ahead
           'mae_at_30': MaskedMAE(at=5),
           'mae_at_60': MaskedMAE(at=11)}

# setup predictor
predictor = Predictor(
    model=stgnn,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=metrics                # metrics to be logged during train/val/test
)

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="logs", name="tsl_intro", version=0)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

trainer = pl.Trainer(max_epochs=1,
                     logger=logger,
                     limit_train_batches=10,  # end an epoch after 10 updates
                     callbacks=[checkpoint_callback])

trainer.fit(predictor, datamodule=dm)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:751: Checkpoint directory C:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\src\logs exists and is not empty.

  | Name          | Type                | Params | Mode 
--------------------------------------------------------------
0 | loss_fn       | MaskedMAE           | 0      | train
1 | train_metrics | MetricCollection    | 0      | train
2 | val_metrics   | MetricCollection    | 0      | train
3 | test_metrics  | MetricCollection    | 0      | train
4 | model         | EncoderDecoderModel | 61.4 K | train
--------------------------------------------------------------
61.4 K    Trainable params
0         Non-trainable params
61.4 K    Total params
0.245     Total estimated model params size (MB)
35        Modules in train mode

Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  7.41it/s]

c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\src\lib\nn\utils.py:12: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:322.)
  emb = emb[[None] * (x.ndim - emb.ndim)]


c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████| 10/10 [00:07<00:00,  1.28it/s, v_num=0, val_mae=8.060, val_mae_at_15=8.020, val_mae_at_30=7.980, val_mae_at_60=8.230, val_mape=0.264, train_mae=8.640, train_mae_at_15=8.340, train_mae_at_30=8.930, train_mae_at_60=9.040, train_mape=0.272]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 10/10 [00:07<00:00,  1.27it/s, v_num=0, val_mae=8.060, val_mae_at_15=8.020, val_mae_at_30=7.980, val_mae_at_60=8.230, val_mape=0.264, train_mae=8.640, train_mae_at_15=8.340, train_mae_at_30=8.930, train_mae_at_60=9.040, train_mape=0.272]


In [ ]:
# predictor.load_model(checkpoint_callback.best_model_path)
# predictor.freeze()

trainer.test(predictor, datamodule=dm)

c:\Users\matya\OneDrive\Bureau\Semester_project\SemesterProject\venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 161/161 [00:17<00:00,  9.00it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            8.070218086242676
        test_mae             8.341201782226562
     test_mae_at_15          8.304732322692871
     test_mae_at_30          8.270994186401367
     test_mae_at_60          8.51406192779541
        test_mape           0.28356271982192993
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_mae': 8.341201782226562,
  'test_mae_at_15': 8.304732322692871,
  'test_mae_at_30': 8.270994186401367,
  'test_mae_at_60': 8.51406192779541,
  'test_mape': 0.28356271982192993,
  'test_loss': 8.070218086242676}]